In [0]:
%pip install openpyxl

In [0]:
dbutils.widgets.removeAll()

In [0]:
%sql
create widget text container default "raw";
create widget text catalogo default "catalog_au";
create widget text esquema default "bronze";
create widget text storageName default "sasmartdata010826ke";
create widget text fileName default "Diccionario de Datos_La Positiva_PERSONA_260429.xlsx";

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")
fileName = dbutils.widgets.get("fileName")

abfss_path = f"abfss://{container}@{storageName}.dfs.core.windows.net/{fileName}"
local_path = f"file:/tmp/{fileName}"
local_raw_path = f"/tmp/{fileName}"

In [0]:
print(f"Copiando {abfss_path} a {local_path}...")
dbutils.fs.cp(abfss_path, local_path)

In [0]:
import pandas as pd
import re
import unicodedata
import os
from pyspark.sql.functions import *
from pyspark.sql.types import *

print(f"Leyendo hoja '1. Conceptos de Negocio' de {local_raw_path}...")
# Saltamos la primera fila ya que contiene la leyenda/documentacion explicativa (skiprows=1)
pdf = pd.read_excel(local_raw_path, sheet_name="1. Conceptos de Negocio", skiprows=[1])

def clean_column_name(col):
    col = str(col).strip().lower()
    # Reemplazar espacios y caracteres especiales por guiones bajos
    col = re.sub(r'[\s\/\(\)\[\]\-\?#\n\r\xa0]+', '_', col)
    col = col.strip('_')
    # Quitar tildes y dieresis
    col = "".join(c for c in unicodedata.normalize('NFD', col) if unicodedata.category(c) != 'Mn')
    return col

pdf.columns = [clean_column_name(c) for c in pdf.columns]
print("Columnas limpiadas:", pdf.columns.tolist())

# Eliminar filas completamente vacias
pdf = pdf.dropna(how='all')

# Reemplazar valores nulos (NaN) para que Spark los reconozca como null y no como string 'nan'
pdf = pdf.where(pd.notnull(pdf), None)
for col_name in pdf.columns:
    pdf[col_name] = pdf[col_name].astype(str).replace('None', None).replace('<NA>', None).replace('nan', None)

In [0]:
spark_df = spark.createDataFrame(pdf)

# Agregar fecha de ingesta
spark_df = spark_df.withColumn("ingestion_date", current_timestamp())

In [0]:
target_table = f"{catalogo}.{esquema}.conceptos_negocio"
dest_cols = spark.table(target_table).columns

# Filtrar las columnas para quedarnos solo con las que existen en la tabla destino
valid_cols = [c for c in spark_df.columns if c in dest_cols]
spark_df_final = spark_df.select(*valid_cols)

# Para cualquier columna faltante en el origen pero presente en el destino, añadirla como nula
for col_name in dest_cols:
    if col_name not in spark_df_final.columns:
        spark_df_final = spark_df_final.withColumn(col_name, lit(None).cast(StringType()))

# Reordenar las columnas para que coincidan exactamente con la firma de la tabla destino
spark_df_final = spark_df_final.select(*dest_cols)

In [0]:
print(f"Escribiendo registros en tabla {target_table}...")
spark_df_final.write.mode("overwrite").insertInto(target_table)

In [0]:
try:
    os.remove(local_raw_path)
    print("Archivo temporal local eliminado.")
except FileNotFoundError:
    print("Archivo ya no existe (eliminado por ejecución paralela). OK.")